In [3]:
import numpy as np
from collections import Counter
from scipy.stats import chi2

from Bio import motifs
from Bio import SeqIO
from Bio.Seq import Seq

In [4]:
for record in SeqIO.parse("/Users/antonkorotkov/Homo_sapiens.GRCh38.dna.chromosome.1.fa", "fasta"):
    print("ID:", record.id)
    print("Description:", record.description)
    print("Length:", len(record.seq))
    print("First 100 bp:", record.seq[:100])

real_seq = record.seq[10000: 20000]

# генерируем случайную посл
counts = Counter(real_seq)
n = len(real_seq)
prob = {nt: counts[nt]/n for nt in 'ACGT'}


np.random.seed(42)
random_seq = ''.join(np.random.choice(list(prob.keys()), size=n, p=list(prob.values())))

ID: 1
Description: 1 dna:chromosome chromosome:GRCh38:1:1:248956422:1 REF
Length: 248956422
First 100 bp: NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN


In [5]:
def dinucleotide_counts(seq):
    """Считает наблюдаемые частоты динуклеотидов Oij"""
    O = np.zeros((4, 4), dtype=int)
    nts = 'ACGT'
    nt_index = {nt: i for i, nt in enumerate(nts)}
    
    for i in range(len(seq)-1):
        O[nt_index[seq[i]], nt_index[seq[i+1]]] += 1
    return O

def expected_counts_zero_order(seq):
    """Считает ожидаемые частоты Eij = N * P(i) * P(j)"""
    n = len(seq) - 1  # количество динуклеотидов
    counts = Counter(seq)
    P = {nt: counts[nt]/len(seq) for nt in 'ACGT'}
    E = np.zeros((4, 4))
    for i, ni in enumerate('ACGT'):
        for j, nj in enumerate('ACGT'):
            E[i, j] = n * P[ni] * P[nj]
    return E

def chi2_statistic(O, E):
    """Вычисляет хи квдрт статистику"""
    chi2_val = np.sum((O - E)**2 / E)
    return chi2_val


In [7]:
# реальная
O_real = dinucleotide_counts(real_seq)
E_real = expected_counts_zero_order(real_seq)
chi2_real = chi2_statistic(O_real, E_real)
p_value_real = 1 - chi2.cdf(chi2_real, df=9)

# сулчайная
O_random = dinucleotide_counts(random_seq)
E_random = expected_counts_zero_order(random_seq)
chi2_random = chi2_statistic(O_random, E_random)
p_value_random = 1 - chi2.cdf(chi2_random, df=9)

# результаты

nts = 'ACGT'

def print_matrix(matrix, name):
    print(f"\n{name}:")
    print("     " + "  ".join(nts))
    for i, row in enumerate(matrix):
        print(f"{nts[i]}  " + "  ".join(f"{val:.2f}" for val in row))

print_matrix(O_real, "Oij (реальная последовательность)")
print_matrix(E_real, "Eij (ожидаемые частоты 0-го порядка)")

print(f"\nχ² (реальная последовательность) = {chi2_real:.2f}")
print(f"p-value = {p_value_real:.4f}")
if p_value_real < 0.05:
    print("Отвергаем гипотезу независимости: нужна модель 1-го порядка")
else:
    print("Не отвергаем гипотезу независимости: 0-й порядок достаточно")

print_matrix(O_random, "Oij (случайная последовательность)")
print_matrix(E_random, "Eij (ожидаемые частоты 0-го порядка)")

print(f"\nχ² (случайная последовательность) = {chi2_random:.2f}")
print(f"p-value = {p_value_random:.4f}")
if p_value_random < 0.05:
    print("Отвергаем гипотезу независимости: нужна модель 1-го порядка")
else:
    print("Не отвергаем гипотезу независимости: 0-й порядок достаточно")


Oij (реальная последовательность):
     A  C  G  T
A  462.00  512.00  805.00  286.00
C  759.00  1090.00  307.00  872.00
G  570.00  869.00  986.00  470.00
T  274.00  557.00  797.00  383.00

Eij (ожидаемые частоты 0-го порядка):
     A  C  G  T
A  426.38  625.22  597.76  415.44
C  625.22  916.79  876.52  609.17
G  597.76  876.52  838.02  582.42
T  415.44  609.17  582.42  404.77

χ² (реальная последовательность) = 862.49
p-value = 0.0000
Отвергаем гипотезу независимости: нужна модель 1-го порядка

Oij (случайная последовательность):
     A  C  G  T
A  429.00  648.00  598.00  436.00
C  651.00  947.00  881.00  595.00
G  613.00  873.00  828.00  530.00
T  418.00  606.00  537.00  409.00

Eij (ожидаемые частоты 0-го порядка):
     A  C  G  T
A  445.59  649.07  600.31  415.83
C  649.07  945.47  874.44  605.71
G  600.31  874.44  808.75  560.21
T  415.83  605.71  560.21  388.05

χ² (случайная последовательность) = 6.32
p-value = 0.7079
Не отвергаем гипотезу независимости: 0-й порядок достаточно
